# Gemini Inference Manifest Generation

This notebook prepares Gemini's transcription manifest for evaluation. It merges raw transcription results with ground-truth segmentations.

**Runtime:** designed to run in a Jupyter kernel whose CWD is `model/colabs/` (so the sibling `common/` package is importable). The `notebook_docker` compose service at `model/notebook_docker/` provides that, with the required `loguru` and `google-cloud-storage` already installed. A stock `Open in Colab` badge would land in a kernel without either, so the badge was removed.

**Note:** This version is focused strictly on data merging and manifest generation. Analysis (WER calculation) and visualization are handled in a separate benchmark notebook.

It performs the following steps:

1.  **Loads existing ground truth** from the baseline manifest.
2.  **Maps Gemini segments** using the `batch_manifest.jsonl` to align raw API results with the benchmark offsets.
3.  **Generates a merged manifest** specifically containing the Gemini predictions.
4.  **Exports the final benchmark** back to Google Cloud Storage.

In [ ]:
# @title Bootstrap and Install Environment
import os

# Clone repository if not already present (for hosted Colab environments)
if not os.path.exists("radio-transcription"):
    !git clone -q https://github.com/watch-duty/radio-transcription.git
else:
    !cd radio-transcription && git pull -q

# Install the model library in editable mode along with required dependencies
try:
    import common

    print("✅ Library 'common' already installed.")
except ImportError:
    print("Installing library and dependencies...")
    %pip install -q -e radio-transcription/model[vertex] loguru tqdm

    import site
    import importlib

    importlib.reload(site)
    print("\n✅ Dependencies installed successfully.")

In [ ]:
import collections
import json
import re
import sys
from pathlib import Path
from typing import Any

from google.cloud import storage
from google.colab import auth, userdata
from loguru import logger

# @markdown ### GCP Configuration from Secrets
GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

# @markdown ### 1. Source Predictions (Choice 2: Auto-Constructed)
# @markdown To locate your `predictions.jsonl`, enter the exact 3 parameters used in your transcription job:
# @markdown *Formula: `gs://{GCS_BUCKET}/{TRANSCRIPTS_BASE_DIR}/{MODEL_ID}/{EXPERIMENT_NAME}/predictions.jsonl`*
TRANSCRIPTS_BASE_DIR = ""  # @param {type:"string"}
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}
EXPERIMENT_NAME = ""  # @param {type:"string"}

# @markdown ### 2. Benchmark Reference & Mapping
# @markdown Reference file containing ground-truth text for WER evaluation (accepts full `gs://` URI or relative path):
BENCHMARK_GROUND_TRUTH_URI = ""  # @param {type:"string"}
# @markdown *(Optional)* Only needed for legacy v1 datasets if timestamp-to-segment mapping lives in a separate file. Defaults to ground truth above if left blank:
BATCH_MANIFEST_URI = ""  # @param {type:"string"}

assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be in Colab secrets."
assert GCS_BUCKET, "GCS_BUCKET must be in Colab secrets."
assert TRANSCRIPTS_BASE_DIR, "TRANSCRIPTS_BASE_DIR must be provided."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided."
assert BENCHMARK_GROUND_TRUTH_URI, (
    "BENCHMARK_GROUND_TRUTH_URI must be provided."
)


# Universal Path Normalization
def _normalize_gcs_uri(path_or_uri: str, bucket_name: str) -> str:
    path_or_uri = path_or_uri.strip()
    if not path_or_uri:
        return ""
    if path_or_uri.startswith("gs://"):
        return path_or_uri
    clean_path = path_or_uri.lstrip("/")
    return f"gs://{bucket_name}/{clean_path}"


BENCHMARK_MANIFEST_URI = _normalize_gcs_uri(
    BENCHMARK_GROUND_TRUTH_URI, GCS_BUCKET
)
MANIFEST_URI = _normalize_gcs_uri(
    BATCH_MANIFEST_URI or BENCHMARK_GROUND_TRUTH_URI, GCS_BUCKET
)

# Clean prefix
_clean_prefix = TRANSCRIPTS_BASE_DIR.strip()
if _clean_prefix.startswith("gs://"):
    _clean_prefix = _clean_prefix.replace(f"gs://{GCS_BUCKET}/", "")
_clean_prefix = _clean_prefix.strip("/")

MODEL_VERSION = re.sub(r"[-\.]", "_", MODEL_ID)
PREDICTIONS_FILE_GCS_PATH = (
    f"{_clean_prefix}/{MODEL_VERSION}/{EXPERIMENT_NAME}/predictions.jsonl"
)

# Clean manifest destination (strip leading 'transcripts/' if present to avoid nesting in inference_manifests/)
_manifest_dest_prefix = re.sub(r"^transcripts/", "", _clean_prefix)
GCS_FINAL_OUTPUT_DIR = (
    f"inference_manifests/{_manifest_dest_prefix}/{MODEL_VERSION}"
)

# Local Filenames
BATCH_MANIFEST_FILENAME = "batch_manifest.jsonl"
EXISTING_BENCHMARK_FILENAME = "benchmark_transcriptions.json"
UPDATED_BENCHMARK_FILENAME = f"{EXPERIMENT_NAME}.jsonl"

# Authenticate to GCS
auth.authenticate_user()
!gcloud config set project {GCP_PROJECT_ID} --quiet

from common.manifest import load_manifest

# Setup logger
logger.remove()
logger.add(sys.stderr, format="<level>{level}</level>: {message}")
print("=" * 60)
print(f"✅ Predictions Source:  gs://{GCS_BUCKET}/{PREDICTIONS_FILE_GCS_PATH}")
print(f"✅ Benchmark Reference: {BENCHMARK_MANIFEST_URI}")
print(f"✅ Batch Manifest:      {MANIFEST_URI}")
print(
    f"✅ Final Manifest Dest: gs://{GCS_BUCKET}/{GCS_FINAL_OUTPUT_DIR}/{UPDATED_BENCHMARK_FILENAME}"
)
print("=" * 60)

In [ ]:
# Global variable to store predictions for diagnostics
gemini_predictions = {}


def merge_gcs_results_to_manifest(
    baseline_data: list[dict[str, Any]],
    batch_manifest_data: list[dict[str, Any]],
    gcs_bucket_name: str,
    output_file: str,
    predictions_gcs_path: str,
) -> dict[str, Any]:
    global gemini_predictions
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(gcs_bucket_name)
    gemini_predictions.clear()

    blob = bucket.blob(predictions_gcs_path)

    if not blob.exists() or (blob.size is not None and blob.size == 0):
        logger.warning(
            f"Predictions file missing or empty: {predictions_gcs_path}"
        )
        return {
            "total": len(baseline_data),
            "matched": 0,
            "missing": len(baseline_data),
        }

    local_preds = "temp_predictions.jsonl"
    blob.download_to_filename(local_preds)

    with open(local_preds) as f:
        for line in f:
            if not line.strip():
                continue
            try:
                data = json.loads(line)
                req_contents = data.get("request", {}).get("contents", [])
                file_uri = ""
                for p in (
                    req_contents[0].get("parts", []) if req_contents else []
                ):
                    if p.get("file_data"):
                        file_uri = p["file_data"]["file_uri"]
                        break
                if not file_uri:
                    continue

                stem = Path(file_uri).stem
                if "__seg" not in stem:
                    continue

                parts = stem.rsplit("__seg", 1)
                example_id, seg_key = parts[0], parts[1]

                raw_text = (
                    data.get("response", {})
                    .get("candidates", [{}])[0]
                    .get("content", {})
                    .get("parts", [{}])[0]
                    .get("text", "")
                )
                gemini_predictions[(example_id, seg_key)] = raw_text.strip()
            except Exception:
                continue

    print(
        f"DEBUG: Loaded {len(gemini_predictions)} unique Gemini segments from {predictions_gcs_path}"
    )

    # Map offsets to segment IDs using the batch manifest
    # Keys: example_id -> {rounded_offset: segment_id}
    offset_map = collections.defaultdict(dict)
    for entry in batch_manifest_data:
        eid = entry.get("example_id", "")
        off = round(float(entry.get("offset", 0.0)), 3)
        offset_map[eid][off] = str(entry.get("segment_id", ""))

    merged_records = []
    matched_count = 0
    EPSILON = 0.002  # 2ms tolerance for float offsets

    for i, b_info in enumerate(baseline_data):
        gemini_text = ""
        # 1. Prefer explicit fields if present in segmented manifests, otherwise strip '__seg' from filename stem
        example_id = str(
            b_info.get("example_id")
            or Path(b_info["audio_filepath"]).stem.rsplit("__seg", 1)[0]
        )
        seg_id = str(b_info.get("segment_id") or "")
        # 2. Fall back to offset lookup only if segment_id was missing
        if not seg_id:
            b_offset = round(float(b_info.get("offset", 0.0)), 3)
            seg_id = offset_map.get(example_id, {}).get(b_offset)

        # Try fuzzy match if exact fails
        if not seg_id and example_id in offset_map:
            for m_off, m_sid in offset_map[example_id].items():
                if abs(m_off - b_offset) < EPSILON:
                    seg_id = m_sid
                    break

        if seg_id and (example_id, seg_id) in gemini_predictions:
            gemini_text = gemini_predictions[(example_id, seg_id)]
            matched_count += 1
        elif i < 5:
            avail = list(offset_map.get(example_id, {}).keys())[:5]
            print(
                f"DEBUG FAIL: {example_id} offset {b_offset}. Found SegID: {seg_id}. Available offsets for this file: {avail}"
            )

        merged_records.append(
            {**b_info, f"pred_text_{MODEL_VERSION}": gemini_text}
        )

    with open(output_file, "w", encoding="utf-8") as f_out:
        f_out.writelines(json.dumps(rec) + "\n" for rec in merged_records)

    return {
        "total": len(baseline_data),
        "matched": matched_count,
        "missing": len(baseline_data) - matched_count,
    }


def run_pipeline() -> None:
    client = storage.Client(project=GCP_PROJECT_ID)
    bucket = client.bucket(GCS_BUCKET)

    bench_blob_path = BENCHMARK_MANIFEST_URI.replace(f"gs://{GCS_BUCKET}/", "")
    bucket.blob(bench_blob_path).download_to_filename(
        EXISTING_BENCHMARK_FILENAME
    )

    manifest_blob_path = MANIFEST_URI.replace(f"gs://{GCS_BUCKET}/", "")
    bucket.blob(manifest_blob_path).download_to_filename(
        BATCH_MANIFEST_FILENAME
    )

    baseline_data = load_manifest(EXISTING_BENCHMARK_FILENAME)
    batch_manifest_data = load_manifest(BATCH_MANIFEST_FILENAME)

    stats = merge_gcs_results_to_manifest(
        baseline_data,
        batch_manifest_data,
        GCS_BUCKET,
        UPDATED_BENCHMARK_FILENAME,
        PREDICTIONS_FILE_GCS_PATH,
    )

    if stats["matched"] > 0:
        gcs_output_path = f"{GCS_FINAL_OUTPUT_DIR}/{UPDATED_BENCHMARK_FILENAME}"
        bucket.blob(gcs_output_path).upload_from_filename(
            UPDATED_BENCHMARK_FILENAME
        )
        logger.info(
            f"Successfully matched {stats['matched']} rows out of {stats['total']}. Uploaded to gs://{GCS_BUCKET}/{gcs_output_path}"
        )
    else:
        logger.error(
            f"CRITICAL: No rows matched ({stats['matched']}/{stats['total']}). check debug logs."
        )

In [ ]:
# @title Run Processing
run_pipeline()